In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

DATA_DIR = Path("output")

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed()

In [ ]:
import json

WINDOW_SELECTION_PATH = OUTPUT_DIR / "xgboost_when" / "selected_window.json"
if not WINDOW_SELECTION_PATH.exists():
    raise FileNotFoundError(
        f"Missing window-selection manifest: {WINDOW_SELECTION_PATH}. "
        "Run predictive_maintenance_xgboost_when.ipynb before this notebook."
    )

with WINDOW_SELECTION_PATH.open(encoding="utf-8") as file:
    window_selection = json.load(file)
if not isinstance(window_selection, dict):
    raise TypeError(f"{WINDOW_SELECTION_PATH} must contain a JSON object.")

required_selection_keys = {
    "selected_window_hours",
    "selection_metric",
    "selection_metric_value",
    "selected_dataset_path",
    "window_metrics_path",
    "selection_source",
}
missing_selection_keys = required_selection_keys - set(window_selection)
if missing_selection_keys:
    raise ValueError(
        f"{WINDOW_SELECTION_PATH} is missing keys: " f"{sorted(missing_selection_keys)}"
    )
if window_selection["selection_metric"] != "test_mae_seconds":
    raise ValueError(
        "Expected selection_metric='test_mae_seconds', found "
        f"{window_selection['selection_metric']!r}."
    )
if window_selection["selection_source"] != "stage_1_xgboost_window_comparison":
    raise ValueError(
        "Unexpected selection_source: " f"{window_selection['selection_source']!r}."
    )

try:
    WINDOW_HOURS = float(window_selection["selected_window_hours"])
    selection_metric_value = float(window_selection["selection_metric_value"])
except (TypeError, ValueError) as error:
    raise ValueError(
        "selected_window_hours and selection_metric_value must be numeric."
    ) from error
if (
    not np.isfinite(WINDOW_HOURS)
    or WINDOW_HOURS <= 0
    or not np.isfinite(selection_metric_value)
    or selection_metric_value < 0
):
    raise ValueError(
        "selected_window_hours must be positive and finite, and the MAE "
        "must be finite and non-negative."
    )

WINDOW_LABEL = f"{WINDOW_HOURS:g}h"
RUN_TAG = f"winsize{WINDOW_LABEL}"
selected_when_path = Path(window_selection["selected_dataset_path"])
expected_when_name = f"dataset_{RUN_TAG}_when.csv"
expected_when_path = DATA_DIR / expected_when_name
if selected_when_path.name != expected_when_name:
    raise ValueError(
        f"Manifest window/path mismatch: selected {WINDOW_LABEL}, but "
        f"selected_dataset_path is {selected_when_path}."
    )
if selected_when_path.resolve() != expected_when_path.resolve():
    raise ValueError(
        f"selected_dataset_path must resolve to {expected_when_path}, found "
        f"{selected_when_path}."
    )
if not selected_when_path.exists():
    raise FileNotFoundError(
        f"Selected WHEN dataset does not exist: {selected_when_path}"
    )

window_metrics_path = Path(window_selection["window_metrics_path"])
if not window_metrics_path.exists():
    raise FileNotFoundError(f"Selection metrics do not exist: {window_metrics_path}")

dataset_path = DATA_DIR / f"dataset_{RUN_TAG}_where.csv"
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Missing WHERE dataset matching the selected window: {dataset_path}"
    )

MODEL_OUTPUT_DIR = OUTPUT_DIR / "where_transformer" / RUN_TAG
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    f"Selected {WINDOW_LABEL} from {WINDOW_SELECTION_PATH} "
    f"({window_selection['selection_metric']}={selection_metric_value:.3f})."
)
print(f"Using WHERE dataset: {dataset_path}")
print(f"Writing artifacts to: {MODEL_OUTPUT_DIR}")

In [ ]:
import json

import pandas as pd

where_df = pd.read_csv(
    dataset_path,
    dtype={"fold_id": "int64"},
    converters={
        "window": json.loads,
        "label": json.loads,
    },
)
where_df = where_df.loc[~where_df["is_bg"]].drop(columns="is_bg")

where_df.head()

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

OUTAGE_TYPE_TO_ID = {
    "Planned": 0,
    "Auto": 1,
}


class WhereOutageDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        window = row["window"]
        label = row["label"]

        outage_type = torch.tensor(
            [OUTAGE_TYPE_TO_ID[event["outage_type"]] for event in window],
            dtype=torch.long,
        )
        from_zone_indices = torch.tensor(
            [event["from_zone_index"] for event in window],
            dtype=torch.long,
        )
        to_zone_indices = torch.tensor(
            [event["to_zone_index"] for event in window],
            dtype=torch.long,
        )
        time_interval_index = torch.tensor(
            [event["time_interval_index"] for event in window],
            dtype=torch.long,
        )
        target = torch.tensor(
            [label["from_zone_index"], label["to_zone_index"]],
            dtype=torch.long,
        )

        return {
            "outage_type": outage_type,
            "from_zone_indices": from_zone_indices,
            "to_zone_indices": to_zone_indices,
            "time_interval_index": time_interval_index,
            "target": target,
        }


TRAIN_FOLDS = [0, 1, 2, 3, 4]
TEST_FOLDS = [-1]

train_where_dataset = WhereOutageDataset(
    where_df[where_df["fold_id"].isin(TRAIN_FOLDS)]
)
test_where_dataset = WhereOutageDataset(where_df[where_df["fold_id"].isin(TEST_FOLDS)])

In [ ]:
for i, sample_dict in enumerate(train_where_dataset):
    print(sample_dict)
    if i == 10:
        break

In [ ]:
observed_time_interval_indices = sorted(
    {
        int(event["time_interval_index"])
        for window in where_df["window"]
        for event in window
    }
)
if not observed_time_interval_indices:
    raise ValueError(
        f"The selected {WINDOW_LABEL} WHERE dataset contains no interval indices."
    )

NUM_TIME_INTERVALS = observed_time_interval_indices[-1] + 1
if observed_time_interval_indices != list(range(NUM_TIME_INTERVALS)):
    raise ValueError(
        "Time interval indices must be contiguous and zero-based; found "
        f"{observed_time_interval_indices}."
    )

train_time_interval_indices = {
    int(event["time_interval_index"])
    for window in where_df.loc[where_df["fold_id"].isin(TRAIN_FOLDS), "window"]
    for event in window
}
test_time_interval_indices = {
    int(event["time_interval_index"])
    for window in where_df.loc[where_df["fold_id"].isin(TEST_FOLDS), "window"]
    for event in window
}
if not train_time_interval_indices or not test_time_interval_indices:
    raise ValueError(
        f"The selected {WINDOW_LABEL} train or test split has no interval indices."
    )
if max(train_time_interval_indices | test_time_interval_indices) >= NUM_TIME_INTERVALS:
    raise ValueError("A train/test interval index exceeds the embedding size.")

print(
    f"{NUM_TIME_INTERVALS = }; "
    f"train={sorted(train_time_interval_indices)}, "
    f"test={sorted(test_time_interval_indices)}"
)

# Model definitions for the selected WHERE window

In [ ]:
MODEL_SCALE = 1

ZONE_EMBEDDING_DIM = int(64 * MODEL_SCALE)
OUTAGE_TYPE_EMBEDDING_DIM = int(32 * MODEL_SCALE)
TIME_INTERVAL_EMBEDDING_DIM = int(32 * MODEL_SCALE)
TRANSFORMER_DIM = int(256 * MODEL_SCALE)

DROPOUT = 0.15

In [ ]:
import torch.nn as nn

num_zones = int(
    max(
        where_df["label"]
        .map(lambda x: max(x["from_zone_index"], x["to_zone_index"]))
        .max(),
        max(
            max(event["from_zone_index"], event["to_zone_index"])
            for window in where_df["window"]
            for event in window
        ),
    )
    + 1
)
num_outage_types = len(OUTAGE_TYPE_TO_ID)


class WhereTransformer(nn.Module):
    """ "Vanilla transformer model"""

    def __init__(
        self,
        num_zones,
        num_outage_types,
        num_time_intervals,
        zone_embedding_dim=ZONE_EMBEDDING_DIM,
        outage_type_embedding_dim=OUTAGE_TYPE_EMBEDDING_DIM,
        time_interval_embedding_dim=TIME_INTERVAL_EMBEDDING_DIM,
        transformer_dim=TRANSFORMER_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.num_zones = num_zones
        self.num_time_intervals = int(num_time_intervals)
        self.zone_embedding = nn.Embedding(num_zones, zone_embedding_dim)
        self.outage_type_embedding = nn.Embedding(
            num_outage_types,
            outage_type_embedding_dim,
        )
        self.time_interval_embedding = nn.Embedding(
            self.num_time_intervals,
            time_interval_embedding_dim,
        )
        self.input_fc = nn.Linear(
            2 * zone_embedding_dim
            + outage_type_embedding_dim
            + time_interval_embedding_dim,
            transformer_dim,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=transformer_dim // 64,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1,
        )
        self.projection = nn.Linear(transformer_dim, 2 * num_zones)

    def forward(
        self,
        from_zone_indices,
        to_zone_indices,
        outage_type,
        time_interval_index,
    ):
        from_zone_emb = self.zone_embedding(from_zone_indices)
        to_zone_emb = self.zone_embedding(to_zone_indices)
        outage_type_emb = self.outage_type_embedding(outage_type)
        time_interval_emb = self.time_interval_embedding(time_interval_index)

        x = torch.cat(
            [from_zone_emb, to_zone_emb, outage_type_emb, time_interval_emb],
            dim=-1,
        )
        x = self.input_fc(x)

        x = self.transformer_encoder(x)

        # Pool the last event in this unpadded, one-sample sequence.
        pooled = x[:, -1, :]

        logits = self.projection(pooled)
        return logits.view(-1, 2, self.num_zones)

In [ ]:
class GraphDistanceUnorderedZoneLoss(nn.Module):
    def __init__(
        self,
        graph,
        num_zones,
        distance_weight=0.2,
        unreachable_distance=None,
        reduction="mean",
    ):
        super().__init__()
        self.num_zones = num_zones
        self.distance_weight = distance_weight
        self.reduction = reduction

        distance_matrix = torch.as_tensor(
            graph.distances(
                source=range(num_zones),
                target=range(num_zones),
                mode="ALL",
            ),
            dtype=torch.float32,
        )
        finite_distances = distance_matrix[torch.isfinite(distance_matrix)]
        if unreachable_distance is None:
            unreachable_distance = finite_distances.max().item() + 1.0
        distance_matrix = torch.where(
            torch.isfinite(distance_matrix),
            distance_matrix,
            torch.full_like(distance_matrix, float(unreachable_distance)),
        )
        self.register_buffer("distance_matrix", distance_matrix)

    def _ordered_loss(self, from_logits, to_logits, target_from, target_to):
        ce_loss = nn.functional.cross_entropy(
            from_logits,
            target_from,
            reduction="none",
        ) + nn.functional.cross_entropy(
            to_logits,
            target_to,
            reduction="none",
        )

        from_probs = nn.functional.softmax(from_logits, dim=-1)
        to_probs = nn.functional.softmax(to_logits, dim=-1)
        from_distance = (from_probs * self.distance_matrix[target_from]).sum(dim=-1)
        to_distance = (to_probs * self.distance_matrix[target_to]).sum(dim=-1)
        distance_loss = from_distance + to_distance

        return ce_loss + self.distance_weight * distance_loss

    def forward(self, logits, target):
        from_logits = logits[:, 0, :]
        to_logits = logits[:, 1, :]
        target_from = target[:, 0]
        target_to = target[:, 1]

        forward_loss = self._ordered_loss(
            from_logits,
            to_logits,
            target_from,
            target_to,
        )
        swapped_loss = self._ordered_loss(
            from_logits,
            to_logits,
            target_to,
            target_from,
        )
        loss = torch.minimum(forward_loss, swapped_loss)

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        if self.reduction == "none":
            return loss
        raise ValueError(f"Unsupported reduction: {self.reduction}")

In [ ]:
class SelfAttentivePooling(nn.Module):
    def __init__(self, input_dim, attention_dim=None, dropout=DROPOUT):
        super().__init__()
        attention_dim = attention_dim or input_dim
        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(attention_dim, 1),
        )

    def forward(self, x):
        attention_scores = self.attention(x).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)
        return (x * attention_weights).sum(dim=1)


class WhereTransformerSAP(nn.Module):
    def __init__(
        self,
        num_zones,
        num_outage_types,
        num_time_intervals,
        zone_embedding_dim=ZONE_EMBEDDING_DIM,
        outage_type_embedding_dim=OUTAGE_TYPE_EMBEDDING_DIM,
        time_interval_embedding_dim=TIME_INTERVAL_EMBEDDING_DIM,
        transformer_dim=TRANSFORMER_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.num_zones = num_zones
        self.num_time_intervals = int(num_time_intervals)
        self.zone_embedding = nn.Embedding(num_zones, zone_embedding_dim)
        self.outage_type_embedding = nn.Embedding(
            num_outage_types,
            outage_type_embedding_dim,
        )
        self.time_interval_embedding = nn.Embedding(
            self.num_time_intervals,
            time_interval_embedding_dim,
        )
        self.input_fc = nn.Linear(
            2 * zone_embedding_dim
            + outage_type_embedding_dim
            + time_interval_embedding_dim,
            transformer_dim,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=transformer_dim // 64,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1,
        )
        self.self_attentive_pooling = SelfAttentivePooling(
            transformer_dim,
            dropout=dropout,
        )
        self.projection = nn.Linear(transformer_dim, 2 * num_zones)

    def forward(
        self,
        from_zone_indices,
        to_zone_indices,
        outage_type,
        time_interval_index,
    ):
        from_zone_emb = self.zone_embedding(from_zone_indices)
        to_zone_emb = self.zone_embedding(to_zone_indices)
        outage_type_emb = self.outage_type_embedding(outage_type)
        time_interval_emb = self.time_interval_embedding(time_interval_index)

        x = torch.cat(
            [from_zone_emb, to_zone_emb, outage_type_emb, time_interval_emb],
            dim=-1,
        )
        x = self.input_fc(x)

        x = self.transformer_encoder(x)

        pooled = self.self_attentive_pooling(x)

        logits = self.projection(pooled)
        return logits.view(-1, 2, self.num_zones)

# Cross-entropy last-event model for the selected window

In [ ]:
LEARNING_RATE = 3e-6
NUM_EPOCHS = 64
VIRTUAL_BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sample_to_device(sample, device):
    return {
        "from_zone_indices": sample["from_zone_indices"].unsqueeze(0).to(device),
        "to_zone_indices": sample["to_zone_indices"].unsqueeze(0).to(device),
        "outage_type": sample["outage_type"].unsqueeze(0).to(device),
        "time_interval_index": sample["time_interval_index"].unsqueeze(0).to(device),
        "target": sample["target"].unsqueeze(0).to(device),
    }


def unordered_zone_loss(logits, target):
    from_logits = logits[:, 0, :]
    to_logits = logits[:, 1, :]
    target_from = target[:, 0]
    target_to = target[:, 1]

    forward_loss = nn.functional.cross_entropy(
        from_logits,
        target_from,
    ) + nn.functional.cross_entropy(
        to_logits,
        target_to,
    )
    swapped_loss = nn.functional.cross_entropy(
        from_logits,
        target_to,
    ) + nn.functional.cross_entropy(
        to_logits,
        target_from,
    )
    return torch.minimum(forward_loss, swapped_loss).mean()


def unordered_zone_correct_count(logits, target):
    pred = logits.argmax(dim=-1)
    pred_sorted = torch.sort(pred, dim=1).values
    target_sorted = torch.sort(target, dim=1).values
    return (pred_sorted == target_sorted).all(dim=1).sum().item()


set_seed()
where_transformer = WhereTransformer(
    num_zones=num_zones,
    num_outage_types=num_outage_types,
    num_time_intervals=NUM_TIME_INTERVALS,
)
where_transformer.to(DEVICE)
optimizer = torch.optim.Adam(where_transformer.parameters(), lr=LEARNING_RATE)

training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    where_transformer.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0
    accumulated_samples = 0
    train_order = torch.randperm(len(train_where_dataset)).tolist()

    for sample_idx in train_order:
        batch = sample_to_device(train_where_dataset[sample_idx], DEVICE)
        logits = where_transformer(
            batch["from_zone_indices"],
            batch["to_zone_indices"],
            batch["outage_type"],
            batch["time_interval_index"],
        )
        loss = unordered_zone_loss(logits, batch["target"])
        loss.backward()

        train_loss_sum += loss.item()
        train_correct += unordered_zone_correct_count(logits, batch["target"])
        train_count += 1
        accumulated_samples += 1

        if accumulated_samples == VIRTUAL_BATCH_SIZE:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            accumulated_samples = 0

    if accumulated_samples:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    train_loss = train_loss_sum / train_count
    train_accuracy = train_correct / train_count

    where_transformer.eval()
    test_loss_sum = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for sample_idx in range(len(test_where_dataset)):
            batch = sample_to_device(test_where_dataset[sample_idx], DEVICE)
            logits = where_transformer(
                batch["from_zone_indices"],
                batch["to_zone_indices"],
                batch["outage_type"],
                batch["time_interval_index"],
            )
            loss = unordered_zone_loss(logits, batch["target"])

            test_loss_sum += loss.item()
            test_correct += unordered_zone_correct_count(logits, batch["target"])
            test_count += 1

    test_loss = test_loss_sum / test_count
    test_accuracy = test_correct / test_count

    epoch_metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    }
    training_history.append(epoch_metrics)

    print(
        f"Epoch {epoch:03d} | "
        f"train loss={train_loss:.6f}, train acc={train_accuracy:.4f} | "
        f"test loss={test_loss:.6f}, test acc={test_accuracy:.4f}"
    )

last_event_ce_checkpoint_path = (
    MODEL_OUTPUT_DIR
    / f"where_transformer_cross_entropy_last_event_{WINDOW_LABEL}_final.pt"
)

torch.save(
    {
        "model_state_dict": where_transformer.state_dict(),
        "window_hours": WINDOW_HOURS,
        "window_label": WINDOW_LABEL,
        "dataset": dataset_path.name,
        "window_selection_manifest": str(WINDOW_SELECTION_PATH),
        "window_selection": window_selection,
        "pooling": "last_event",
        "num_zones": num_zones,
        "num_outage_types": num_outage_types,
        "num_time_intervals": NUM_TIME_INTERVALS,
        "zone_embedding_dim": ZONE_EMBEDDING_DIM,
        "outage_type_embedding_dim": OUTAGE_TYPE_EMBEDDING_DIM,
        "time_interval_embedding_dim": TIME_INTERVAL_EMBEDDING_DIM,
        "transformer_dim": TRANSFORMER_DIM,
        "dropout": DROPOUT,
        "outage_type_to_id": OUTAGE_TYPE_TO_ID,
        "training_history": training_history,
    },
    last_event_ce_checkpoint_path,
)

print(
    "Saved final cross-entropy last-event checkpoint to "
    f"{last_event_ce_checkpoint_path}"
)

In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in training_history]
train_losses = [row["train_loss"] for row in training_history]
test_losses = [row["test_loss"] for row in training_history]
train_accuracies = [row["train_accuracy"] for row in training_history]
test_accuracies = [row["test_accuracy"] for row in training_history]

plt.figure(figsize=(8, 4))

plt.plot(epochs, train_losses, label="Train")
plt.plot(epochs, test_losses, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
# plt.title("Training and test loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR
    / f"where_transformer_cross_entropy_last_event_{WINDOW_LABEL}_training_loss.pdf",
    dpi=300,
)
plt.show()

plt.figure(figsize=(8, 4))

plt.plot(epochs, train_accuracies, label="Train")
plt.plot(epochs, test_accuracies, label="Test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
# plt.title("Training and test accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR
    / f"where_transformer_cross_entropy_last_event_{WINDOW_LABEL}_training_accuracy.pdf",
    dpi=300,
)
plt.show()

In [ ]:
from collections import deque
from pathlib import Path

import igraph as ig
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

GRAPH_PATH = Path("res/outage_graph.pkl")
EDGE_ZONE_PATH = Path("output/edge_zones.csv")
RANDOM_PAIR_BASELINE_SIZE = 10_000


def build_zone_distance_lookup(edge_zone_path):
    edge_zone_df = pd.read_csv(edge_zone_path)
    zones = sorted(
        set(edge_zone_df["from_zone"].astype(int))
        | set(edge_zone_df["to_zone"].astype(int))
    )
    adjacency = {zone: {zone} for zone in zones}

    for from_zone, to_zone in zip(
        edge_zone_df["from_zone"].astype(int),
        edge_zone_df["to_zone"].astype(int),
    ):
        adjacency.setdefault(from_zone, {from_zone}).add(to_zone)
        adjacency.setdefault(to_zone, {to_zone}).add(from_zone)

    distances = {}
    for source in adjacency:
        source_distances = {source: 0}
        queue = deque([source])
        while queue:
            current = queue.popleft()
            for neighbor in adjacency[current]:
                if neighbor not in source_distances:
                    source_distances[neighbor] = source_distances[current] + 1
                    queue.append(neighbor)
        distances[source] = source_distances

    return distances


def zone_distance(source_zone, target_zone, distance_lookup):
    return distance_lookup.get(int(source_zone), {}).get(int(target_zone), np.nan)


def unordered_pair_graph_distance(true_pair, pred_pair, distance_lookup):
    direct = (
        zone_distance(true_pair[0], pred_pair[0], distance_lookup)
        + zone_distance(true_pair[1], pred_pair[1], distance_lookup)
    ) / 2
    swapped = (
        zone_distance(true_pair[0], pred_pair[1], distance_lookup)
        + zone_distance(true_pair[1], pred_pair[0], distance_lookup)
    ) / 2
    candidates = [distance for distance in [direct, swapped] if np.isfinite(distance)]
    return min(candidates) if candidates else np.nan


def summarize_pair_distances(distances):
    distances = np.asarray(distances, dtype=float)
    finite = distances[np.isfinite(distances)]
    return {
        "mean_distance": finite.mean(),
        "median_distance": np.median(finite),
        "p90_distance": np.percentile(finite, 90),
        "same_zone_pair_rate": (finite == 0).mean(),
        "adjacent_or_same_pair_rate": (finite <= 1).mean(),
        "unreachable_count": int(np.isnan(distances).sum()),
    }


def collect_where_predictions(model, dataset, device):
    model.eval()
    rows = []
    with torch.no_grad():
        for sample in dataset:
            batch = sample_to_device(sample, device)
            logits = model(
                batch["from_zone_indices"],
                batch["to_zone_indices"],
                batch["outage_type"],
                batch["time_interval_index"],
            )
            pred_pair = logits.argmax(dim=-1).squeeze(0).cpu().tolist()
            true_pair = sample["target"].cpu().tolist()
            rows.append(
                {
                    "true_from_zone": int(true_pair[0]),
                    "true_to_zone": int(true_pair[1]),
                    "pred_from_zone": int(pred_pair[0]),
                    "pred_to_zone": int(pred_pair[1]),
                    "exact_pair_match": sorted(true_pair) == sorted(pred_pair),
                }
            )
    return pd.DataFrame(rows)


outage_graph = ig.Graph.Read_Pickle(str(GRAPH_PATH))
zone_distance_lookup = build_zone_distance_lookup(EDGE_ZONE_PATH)
zone_ids = sorted(zone_distance_lookup)
rng = np.random.default_rng(RANDOM_STATE)

raw_graph_diameter = outage_graph.diameter(directed=False, unconn=True)
node_a, node_b = rng.choice(outage_graph.vcount(), size=2, replace=False)
random_node_distance = outage_graph.distances(
    source=[int(node_a)],
    target=[int(node_b)],
    mode="ALL",
)[0][0]

zone_graph_diameter = max(
    distance
    for source_distances in zone_distance_lookup.values()
    for distance in source_distances.values()
)
zone_a, zone_b = rng.choice(zone_ids, size=2, replace=False)
random_zone_distance = zone_distance_lookup[int(zone_a)][int(zone_b)]

checkpoint_path = Path(last_event_ce_checkpoint_path)
if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Missing selected-window checkpoint: {checkpoint_path}. "
        "Run the cross-entropy last-event training cell first."
    )
try:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

if not np.isclose(float(checkpoint.get("window_hours", np.nan)), WINDOW_HOURS):
    raise ValueError(
        f"Checkpoint window does not match selected {WINDOW_LABEL}: {checkpoint_path}"
    )
if checkpoint.get("dataset") != dataset_path.name:
    raise ValueError(
        f"Checkpoint dataset does not match {dataset_path.name}: {checkpoint_path}"
    )
if checkpoint.get("num_time_intervals") != NUM_TIME_INTERVALS:
    raise ValueError(
        "Checkpoint time-interval vocabulary does not match the selected dataset."
    )
if checkpoint.get("pooling") != "last_event":
    raise ValueError(f"Checkpoint pooling metadata is invalid: {checkpoint_path}")

where_transformer.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded {WINDOW_LABEL} model checkpoint from {checkpoint_path}")

prediction_df = collect_where_predictions(where_transformer, test_where_dataset, DEVICE)
pair_distances = np.asarray(
    [
        unordered_pair_graph_distance(
            (row.true_from_zone, row.true_to_zone),
            (row.pred_from_zone, row.pred_to_zone),
            zone_distance_lookup,
        )
        for row in prediction_df.itertuples(index=False)
    ],
    dtype=float,
)

random_pair_distances = np.asarray(
    [
        unordered_pair_graph_distance(
            tuple(
                prediction_df.iloc[i % len(prediction_df)][
                    ["true_from_zone", "true_to_zone"]
                ].to_numpy()
            ),
            rng.choice(zone_ids, size=2, replace=True),
            zone_distance_lookup,
        )
        for i in range(RANDOM_PAIR_BASELINE_SIZE)
    ],
    dtype=float,
)

model_summary = summarize_pair_distances(pair_distances)
random_summary = summarize_pair_distances(random_pair_distances)
model_summary["exact_pair_accuracy"] = prediction_df["exact_pair_match"].mean()
model_summary["mean_distance_vs_diameter"] = (
    model_summary["mean_distance"] / zone_graph_diameter
)
model_summary["mean_distance_reduction_vs_random"] = 1 - (
    model_summary["mean_distance"] / random_summary["mean_distance"]
)

summary_df = pd.DataFrame(
    [
        {"model": "Transformer", **model_summary},
        {"model": "Random zone-pair baseline", **random_summary},
    ]
)

print(f"Raw outage graph: {outage_graph.vcount()} nodes, {outage_graph.ecount()} edges")
print(f"Maximum raw graph node distance: {raw_graph_diameter}")
print(
    f"Random raw nodes {int(node_a)} -> {int(node_b)} distance: "
    f"{random_node_distance}"
)
print(f"Maximum zone graph distance: {zone_graph_diameter}")
print(f"Random zones {int(zone_a)} -> {int(zone_b)} distance: {random_zone_distance}")
print(
    "Average unordered zone-pair distance, model: "
    f"{model_summary['mean_distance']:.3f}"
)
print(
    "Average unordered zone-pair distance, random baseline: "
    f"{random_summary['mean_distance']:.3f}"
)
print(
    "Model mean unordered pair distance: "
    f"{model_summary['mean_distance']:.3f} / {zone_graph_diameter} "
    f"({model_summary['mean_distance_vs_diameter']:.1%} of zone diameter)"
)
print(
    "Distance reduction versus random zone-pair baseline: "
    f"{model_summary['mean_distance_reduction_vs_random']:.1%}"
)

display(summary_df.round(3))
display(prediction_df.head())

bins = np.arange(-0.5, zone_graph_diameter + 1.5, 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    pair_distances[np.isfinite(pair_distances)], bins=bins, alpha=0.7, label="Model"
)
ax.hist(
    random_pair_distances[np.isfinite(random_pair_distances)],
    bins=bins,
    histtype="step",
    linewidth=2,
    label="Random baseline",
)
ax.set_xlabel("Unordered zone-pair graph distance")
ax.set_ylabel("Count")
# ax.set_title("Where-model graph-distance error")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR
    / f"where_transformer_cross_entropy_last_event_{WINDOW_LABEL}_graph_distance_error.pdf",
    dpi=300,
)
plt.show()

# Cross-entropy self-attentive-pooling model for the selected window

In [ ]:
LEARNING_RATE = 3e-6
NUM_EPOCHS = 64
VIRTUAL_BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sample_to_device(sample, device):
    return {
        "from_zone_indices": sample["from_zone_indices"].unsqueeze(0).to(device),
        "to_zone_indices": sample["to_zone_indices"].unsqueeze(0).to(device),
        "outage_type": sample["outage_type"].unsqueeze(0).to(device),
        "time_interval_index": sample["time_interval_index"].unsqueeze(0).to(device),
        "target": sample["target"].unsqueeze(0).to(device),
    }


def unordered_zone_loss(logits, target):
    from_logits = logits[:, 0, :]
    to_logits = logits[:, 1, :]
    target_from = target[:, 0]
    target_to = target[:, 1]

    forward_loss = nn.functional.cross_entropy(
        from_logits,
        target_from,
    ) + nn.functional.cross_entropy(
        to_logits,
        target_to,
    )
    swapped_loss = nn.functional.cross_entropy(
        from_logits,
        target_to,
    ) + nn.functional.cross_entropy(
        to_logits,
        target_from,
    )
    return torch.minimum(forward_loss, swapped_loss).mean()


def unordered_zone_correct_count(logits, target):
    pred = logits.argmax(dim=-1)
    pred_sorted = torch.sort(pred, dim=1).values
    target_sorted = torch.sort(target, dim=1).values
    return (pred_sorted == target_sorted).all(dim=1).sum().item()


set_seed()
where_transformer_sap = WhereTransformerSAP(
    num_zones=num_zones,
    num_outage_types=num_outage_types,
    num_time_intervals=NUM_TIME_INTERVALS,
)
where_transformer_sap
where_transformer_sap.to(DEVICE)
optimizer = torch.optim.Adam(where_transformer_sap.parameters(), lr=LEARNING_RATE)

training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    where_transformer_sap.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0
    accumulated_samples = 0
    train_order = torch.randperm(len(train_where_dataset)).tolist()

    for sample_idx in train_order:
        batch = sample_to_device(train_where_dataset[sample_idx], DEVICE)
        logits = where_transformer_sap(
            batch["from_zone_indices"],
            batch["to_zone_indices"],
            batch["outage_type"],
            batch["time_interval_index"],
        )
        loss = unordered_zone_loss(logits, batch["target"])
        loss.backward()

        train_loss_sum += loss.item()
        train_correct += unordered_zone_correct_count(logits, batch["target"])
        train_count += 1
        accumulated_samples += 1

        if accumulated_samples == VIRTUAL_BATCH_SIZE:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            accumulated_samples = 0

    if accumulated_samples:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    train_loss = train_loss_sum / train_count
    train_accuracy = train_correct / train_count

    where_transformer_sap.eval()
    test_loss_sum = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for sample_idx in range(len(test_where_dataset)):
            batch = sample_to_device(test_where_dataset[sample_idx], DEVICE)
            logits = where_transformer_sap(
                batch["from_zone_indices"],
                batch["to_zone_indices"],
                batch["outage_type"],
                batch["time_interval_index"],
            )
            loss = unordered_zone_loss(logits, batch["target"])

            test_loss_sum += loss.item()
            test_correct += unordered_zone_correct_count(logits, batch["target"])
            test_count += 1

    test_loss = test_loss_sum / test_count
    test_accuracy = test_correct / test_count

    epoch_metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    }
    training_history.append(epoch_metrics)

    print(
        f"Epoch {epoch:03d} | "
        f"train loss={train_loss:.6f}, train acc={train_accuracy:.4f} | "
        f"test loss={test_loss:.6f}, test acc={test_accuracy:.4f}"
    )

sap_ce_checkpoint_path = (
    MODEL_OUTPUT_DIR / f"where_transformer_cross_entropy_sap_{WINDOW_LABEL}_final.pt"
)

torch.save(
    {
        "model_state_dict": where_transformer_sap.state_dict(),
        "window_hours": WINDOW_HOURS,
        "window_label": WINDOW_LABEL,
        "dataset": dataset_path.name,
        "window_selection_manifest": str(WINDOW_SELECTION_PATH),
        "window_selection": window_selection,
        "pooling": "self_attentive",
        "num_zones": num_zones,
        "num_outage_types": num_outage_types,
        "num_time_intervals": NUM_TIME_INTERVALS,
        "zone_embedding_dim": ZONE_EMBEDDING_DIM,
        "outage_type_embedding_dim": OUTAGE_TYPE_EMBEDDING_DIM,
        "time_interval_embedding_dim": TIME_INTERVAL_EMBEDDING_DIM,
        "transformer_dim": TRANSFORMER_DIM,
        "dropout": DROPOUT,
        "outage_type_to_id": OUTAGE_TYPE_TO_ID,
        "training_history": training_history,
    },
    sap_ce_checkpoint_path,
)

print(
    "Saved final cross-entropy self-attentive-pooling checkpoint to "
    f"{sap_ce_checkpoint_path}"
)

# Adjacency-aware last-event model for the selected window

In [ ]:
ADJACENCY_DISTANCE_WEIGHT = 2.0

edge_zone_df = pd.read_csv(EDGE_ZONE_PATH)
zone_edges = [
    (int(from_zone), int(to_zone))
    for from_zone, to_zone in zip(edge_zone_df["from_zone"], edge_zone_df["to_zone"])
    if 0 <= int(from_zone) < num_zones and 0 <= int(to_zone) < num_zones
]
zone_graph = ig.Graph(n=num_zones, edges=zone_edges, directed=False)
zone_graph.simplify(multiple=True, loops=True)

set_seed()
adjacency_where_transformer = WhereTransformer(
    num_zones=num_zones,
    num_outage_types=num_outage_types,
    num_time_intervals=NUM_TIME_INTERVALS,
)
adjacency_where_transformer.to(DEVICE)

adjacency_criterion = GraphDistanceUnorderedZoneLoss(
    zone_graph,
    num_zones=num_zones,
    distance_weight=ADJACENCY_DISTANCE_WEIGHT,
).to(DEVICE)
adjacency_optimizer = torch.optim.Adam(
    adjacency_where_transformer.parameters(),
    lr=LEARNING_RATE,
)

adjacency_training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    adjacency_where_transformer.train()
    adjacency_optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0
    accumulated_samples = 0
    train_order = torch.randperm(len(train_where_dataset)).tolist()

    for sample_idx in train_order:
        batch = sample_to_device(train_where_dataset[sample_idx], DEVICE)
        logits = adjacency_where_transformer(
            batch["from_zone_indices"],
            batch["to_zone_indices"],
            batch["outage_type"],
            batch["time_interval_index"],
        )
        loss = adjacency_criterion(logits, batch["target"])
        loss.backward()

        train_loss_sum += loss.item()
        train_correct += unordered_zone_correct_count(logits, batch["target"])
        train_count += 1
        accumulated_samples += 1

        if accumulated_samples == VIRTUAL_BATCH_SIZE:
            adjacency_optimizer.step()
            adjacency_optimizer.zero_grad(set_to_none=True)
            accumulated_samples = 0

    if accumulated_samples:
        adjacency_optimizer.step()
        adjacency_optimizer.zero_grad(set_to_none=True)

    train_loss = train_loss_sum / train_count
    train_accuracy = train_correct / train_count

    adjacency_where_transformer.eval()
    test_loss_sum = 0.0
    test_cross_entropy_loss_sum = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for sample_idx in range(len(test_where_dataset)):
            batch = sample_to_device(test_where_dataset[sample_idx], DEVICE)
            logits = adjacency_where_transformer(
                batch["from_zone_indices"],
                batch["to_zone_indices"],
                batch["outage_type"],
                batch["time_interval_index"],
            )
            loss = adjacency_criterion(logits, batch["target"])
            cross_entropy_loss = unordered_zone_loss(logits, batch["target"])

            test_loss_sum += loss.item()
            test_cross_entropy_loss_sum += cross_entropy_loss.item()
            test_correct += unordered_zone_correct_count(logits, batch["target"])
            test_count += 1

    test_loss = test_loss_sum / test_count
    test_cross_entropy_loss = test_cross_entropy_loss_sum / test_count
    test_accuracy = test_correct / test_count

    epoch_metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "test_loss": test_loss,
        "test_cross_entropy_loss": test_cross_entropy_loss,
        "test_accuracy": test_accuracy,
    }
    adjacency_training_history.append(epoch_metrics)

    print(
        f"Epoch {epoch:03d} | "
        f"train loss={train_loss:.6f}, train acc={train_accuracy:.4f} | "
        f"test adjacency loss={test_loss:.6f}, "
        f"test CE loss={test_cross_entropy_loss:.6f}, "
        f"test acc={test_accuracy:.4f}"
    )

last_event_adjacency_checkpoint_path = (
    MODEL_OUTPUT_DIR
    / f"where_transformer_adjacency_aware_last_event_{WINDOW_LABEL}_final.pt"
)

torch.save(
    {
        "model_state_dict": adjacency_where_transformer.state_dict(),
        "window_hours": WINDOW_HOURS,
        "window_label": WINDOW_LABEL,
        "dataset": dataset_path.name,
        "window_selection_manifest": str(WINDOW_SELECTION_PATH),
        "window_selection": window_selection,
        "pooling": "last_event",
        "num_zones": num_zones,
        "num_outage_types": num_outage_types,
        "num_time_intervals": NUM_TIME_INTERVALS,
        "zone_embedding_dim": ZONE_EMBEDDING_DIM,
        "outage_type_embedding_dim": OUTAGE_TYPE_EMBEDDING_DIM,
        "time_interval_embedding_dim": TIME_INTERVAL_EMBEDDING_DIM,
        "transformer_dim": TRANSFORMER_DIM,
        "dropout": DROPOUT,
        "outage_type_to_id": OUTAGE_TYPE_TO_ID,
        "distance_weight": ADJACENCY_DISTANCE_WEIGHT,
        "training_history": adjacency_training_history,
    },
    last_event_adjacency_checkpoint_path,
)

print(
    "Saved final adjacency-aware last-event checkpoint to "
    f"{last_event_adjacency_checkpoint_path}"
)

# Adjacency-aware self-attentive-pooling model for the selected window

In [ ]:
ADJACENCY_DISTANCE_WEIGHT = 2.0

edge_zone_df = pd.read_csv(EDGE_ZONE_PATH)
zone_edges = [
    (int(from_zone), int(to_zone))
    for from_zone, to_zone in zip(edge_zone_df["from_zone"], edge_zone_df["to_zone"])
    if 0 <= int(from_zone) < num_zones and 0 <= int(to_zone) < num_zones
]
zone_graph = ig.Graph(n=num_zones, edges=zone_edges, directed=False)
zone_graph.simplify(multiple=True, loops=True)

set_seed()
adjacency_where_transformer_sap = WhereTransformerSAP(
    num_zones=num_zones,
    num_outage_types=num_outage_types,
    num_time_intervals=NUM_TIME_INTERVALS,
)
adjacency_where_transformer_sap.to(DEVICE)

adjacency_criterion = GraphDistanceUnorderedZoneLoss(
    zone_graph,
    num_zones=num_zones,
    distance_weight=ADJACENCY_DISTANCE_WEIGHT,
).to(DEVICE)
adjacency_optimizer = torch.optim.Adam(
    adjacency_where_transformer_sap.parameters(),
    lr=LEARNING_RATE,
)

adjacency_training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    adjacency_where_transformer_sap.train()
    adjacency_optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0
    accumulated_samples = 0
    train_order = torch.randperm(len(train_where_dataset)).tolist()

    for sample_idx in train_order:
        batch = sample_to_device(train_where_dataset[sample_idx], DEVICE)
        logits = adjacency_where_transformer_sap(
            batch["from_zone_indices"],
            batch["to_zone_indices"],
            batch["outage_type"],
            batch["time_interval_index"],
        )
        loss = adjacency_criterion(logits, batch["target"])
        loss.backward()

        train_loss_sum += loss.item()
        train_correct += unordered_zone_correct_count(logits, batch["target"])
        train_count += 1
        accumulated_samples += 1

        if accumulated_samples == VIRTUAL_BATCH_SIZE:
            adjacency_optimizer.step()
            adjacency_optimizer.zero_grad(set_to_none=True)
            accumulated_samples = 0

    if accumulated_samples:
        adjacency_optimizer.step()
        adjacency_optimizer.zero_grad(set_to_none=True)

    train_loss = train_loss_sum / train_count
    train_accuracy = train_correct / train_count

    adjacency_where_transformer_sap.eval()
    test_loss_sum = 0.0
    test_cross_entropy_loss_sum = 0.0
    test_correct = 0
    test_count = 0

    with torch.no_grad():
        for sample_idx in range(len(test_where_dataset)):
            batch = sample_to_device(test_where_dataset[sample_idx], DEVICE)
            logits = adjacency_where_transformer_sap(
                batch["from_zone_indices"],
                batch["to_zone_indices"],
                batch["outage_type"],
                batch["time_interval_index"],
            )
            loss = adjacency_criterion(logits, batch["target"])
            cross_entropy_loss = unordered_zone_loss(logits, batch["target"])

            test_loss_sum += loss.item()
            test_cross_entropy_loss_sum += cross_entropy_loss.item()
            test_correct += unordered_zone_correct_count(logits, batch["target"])
            test_count += 1

    test_loss = test_loss_sum / test_count
    test_cross_entropy_loss = test_cross_entropy_loss_sum / test_count
    test_accuracy = test_correct / test_count

    epoch_metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "test_loss": test_loss,
        "test_cross_entropy_loss": test_cross_entropy_loss,
        "test_accuracy": test_accuracy,
    }
    adjacency_training_history.append(epoch_metrics)

    print(
        f"Epoch {epoch:03d} | "
        f"train loss={train_loss:.6f}, train acc={train_accuracy:.4f} | "
        f"test adjacency loss={test_loss:.6f}, "
        f"test CE loss={test_cross_entropy_loss:.6f}, "
        f"test acc={test_accuracy:.4f}"
    )

sap_adjacency_checkpoint_path = (
    MODEL_OUTPUT_DIR / f"where_transformer_adjacency_aware_sap_{WINDOW_LABEL}_final.pt"
)

torch.save(
    {
        "model_state_dict": adjacency_where_transformer_sap.state_dict(),
        "window_hours": WINDOW_HOURS,
        "window_label": WINDOW_LABEL,
        "dataset": dataset_path.name,
        "window_selection_manifest": str(WINDOW_SELECTION_PATH),
        "window_selection": window_selection,
        "pooling": "self_attentive",
        "num_zones": num_zones,
        "num_outage_types": num_outage_types,
        "num_time_intervals": NUM_TIME_INTERVALS,
        "zone_embedding_dim": ZONE_EMBEDDING_DIM,
        "outage_type_embedding_dim": OUTAGE_TYPE_EMBEDDING_DIM,
        "time_interval_embedding_dim": TIME_INTERVAL_EMBEDDING_DIM,
        "transformer_dim": TRANSFORMER_DIM,
        "dropout": DROPOUT,
        "outage_type_to_id": OUTAGE_TYPE_TO_ID,
        "distance_weight": ADJACENCY_DISTANCE_WEIGHT,
        "training_history": adjacency_training_history,
    },
    sap_adjacency_checkpoint_path,
)

print(
    "Saved final adjacency-aware self-attentive-pooling checkpoint to "
    f"{sap_adjacency_checkpoint_path}"
)

In [ ]:
baseline_prediction_df = collect_where_predictions(
    where_transformer,
    test_where_dataset,
    DEVICE,
)

baseline_sap_df = collect_where_predictions(
    where_transformer_sap,
    test_where_dataset,
    DEVICE,
)

adjacency_prediction_df = collect_where_predictions(
    adjacency_where_transformer,
    test_where_dataset,
    DEVICE,
)

adjacency_prediction_sap_df = collect_where_predictions(
    adjacency_where_transformer_sap,
    test_where_dataset,
    DEVICE,
)


baseline_pair_distances = np.asarray(
    [
        unordered_pair_graph_distance(
            (row.true_from_zone, row.true_to_zone),
            (row.pred_from_zone, row.pred_to_zone),
            zone_distance_lookup,
        )
        for row in baseline_prediction_df.itertuples(index=False)
    ],
    dtype=float,
)
baseline_pair_distances_sap = np.asarray(
    [
        unordered_pair_graph_distance(
            (row.true_from_zone, row.true_to_zone),
            (row.pred_from_zone, row.pred_to_zone),
            zone_distance_lookup,
        )
        for row in baseline_sap_df.itertuples(index=False)
    ],
    dtype=float,
)
adjacency_pair_distances = np.asarray(
    [
        unordered_pair_graph_distance(
            (row.true_from_zone, row.true_to_zone),
            (row.pred_from_zone, row.pred_to_zone),
            zone_distance_lookup,
        )
        for row in adjacency_prediction_df.itertuples(index=False)
    ],
    dtype=float,
)
adjacency_pair_distances_sap = np.asarray(
    [
        unordered_pair_graph_distance(
            (row.true_from_zone, row.true_to_zone),
            (row.pred_from_zone, row.pred_to_zone),
            zone_distance_lookup,
        )
        for row in adjacency_prediction_sap_df.itertuples(index=False)
    ],
    dtype=float,
)


baseline_summary = summarize_pair_distances(baseline_pair_distances)
baseline_summary["exact_pair_accuracy"] = baseline_prediction_df[
    "exact_pair_match"
].mean()

baseline_summary_sap = summarize_pair_distances(baseline_pair_distances_sap)
baseline_summary_sap["exact_pair_accuracy"] = baseline_sap_df["exact_pair_match"].mean()

adjacency_summary = summarize_pair_distances(adjacency_pair_distances)
adjacency_summary["exact_pair_accuracy"] = adjacency_prediction_df[
    "exact_pair_match"
].mean()
adjacency_summary["mean_distance_reduction_vs_cross_entropy"] = 1 - (
    adjacency_summary["mean_distance"] / baseline_summary["mean_distance"]
)

adjacency_summary_sap = summarize_pair_distances(adjacency_pair_distances_sap)
adjacency_summary_sap["exact_pair_accuracy"] = adjacency_prediction_sap_df[
    "exact_pair_match"
].mean()
adjacency_summary_sap["mean_distance_reduction_vs_cross_entropy"] = 1 - (
    adjacency_summary_sap["mean_distance"] / baseline_summary_sap["mean_distance"]
)

adjacency_comparison_df = pd.DataFrame(
    [
        {"model": "Cross entropy", **baseline_summary},
        {"model": "Cross entropy (SAP)", **baseline_summary_sap},
        {"model": "Adjacency-aware", **adjacency_summary},
        {"model": "Adjacency-aware (SAP)", **adjacency_summary_sap},
    ]
)

comparison_table_path = (
    MODEL_OUTPUT_DIR / f"where_transformer_{WINDOW_LABEL}_model_comparison.csv"
)
adjacency_comparison_df.to_csv(comparison_table_path, index=False)
print(f"Saved {WINDOW_LABEL} model comparison to {comparison_table_path}")

display(adjacency_comparison_df.round(3))
print(
    "Adjacency-aware mean unordered pair distance reduction vs cross entropy: "
    f"{adjacency_summary['mean_distance_reduction_vs_cross_entropy']:.1%}"
)
print(
    "Adjacency-aware (SAP) mean unordered pair distance reduction vs cross entropy: "
    f"{adjacency_summary_sap['mean_distance_reduction_vs_cross_entropy']:.1%}"
)

bins = np.arange(-0.5, zone_graph_diameter + 1.5, 1)
bin_centers = (bins[:-1] + bins[1:]) / 2
bar_width = 0.2
baseline_counts, _ = np.histogram(
    baseline_pair_distances[np.isfinite(baseline_pair_distances)], bins=bins
)
baseline_sap_counts, _ = np.histogram(
    baseline_pair_distances_sap[np.isfinite(baseline_pair_distances_sap)], bins=bins
)
adjacency_counts, _ = np.histogram(
    adjacency_pair_distances[np.isfinite(adjacency_pair_distances)], bins=bins
)
adjacency_sap_counts, _ = np.histogram(
    adjacency_pair_distances_sap[np.isfinite(adjacency_pair_distances_sap)], bins=bins
)
baseline_percentages = 100 * baseline_counts / baseline_counts.sum()
baseline_sap_percentages = 100 * baseline_sap_counts / baseline_sap_counts.sum()
adjacency_percentages = 100 * adjacency_counts / adjacency_counts.sum()
adjacency_sap_percentages = 100 * adjacency_sap_counts / adjacency_sap_counts.sum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    bin_centers - 1.5 * bar_width,
    baseline_percentages,
    width=bar_width,
    label="Cross entropy",
    align="center",
)
ax.bar(
    bin_centers - 0.5 * bar_width,
    baseline_sap_percentages,
    width=bar_width,
    label="Cross entropy (SAP)",
    align="center",
)
ax.bar(
    bin_centers + 0.5 * bar_width,
    adjacency_percentages,
    width=bar_width,
    label="Adjacency-aware",
    align="center",
)
ax.bar(
    bin_centers + 1.5 * bar_width,
    adjacency_sap_percentages,
    width=bar_width,
    label="Adjacency-aware (SAP)",
    align="center",
)
ax.set_xticks(bin_centers)
ax.set_xlabel("Unordered zone-pair graph distance")
ax.set_ylabel("Percentage (%)")
# ax.set_title("Cross-entropy vs adjacency-aware graph-distance error")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR
    / f"where_transformer_{WINDOW_LABEL}_cross_entropy_vs_adjacency_graph_distance_error.pdf",
    dpi=300,
)
plt.show()

# Selected-window paper plot

In [ ]:
paper_bar_width = 0.36

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    bin_centers - paper_bar_width / 2,
    baseline_percentages,
    width=paper_bar_width,
    label="Cross entropy",
    align="center",
)
ax.bar(
    bin_centers + paper_bar_width / 2,
    adjacency_percentages,
    width=paper_bar_width,
    label="Adjacency-aware",
    align="center",
)
ax.set_xticks(bin_centers)
ax.set_xlabel("Unordered zone-pair graph distance")
ax.set_ylabel("Percentage (%)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(
    MODEL_OUTPUT_DIR
    / f"where_transformer_{WINDOW_LABEL}_cross_entropy_vs_adjacency_graph_distance_error_paper.pdf",
    dpi=300,
)
plt.show()